# Story Category Version 1

This notebook creates a first version of the three-category reference dataset using only high-confidence StoryWeaver categories.

Version 1 mapping:

- `Animal Stories` -> `Animals`
- `Family & Friends` + `Growing Up` -> `Daily Life`
- `STEM` + `Non-fiction` -> `Science & Knowledge`

LIDA stories are not assigned here yet. They will be classified later.

In [1]:
from pathlib import Path
import json

import pandas as pd

cwd = Path.cwd()
ROOT = cwd if (cwd / "data").exists() else cwd.parent

MERGE_READY_PATH = ROOT / "data/processed/lida_storyweaver_merge_ready.json"
CATEGORY_V1_OUTPUT_PATH = ROOT / "data/processed/story_categories_v1.json"

print("Project root:", ROOT)
print("Merge-ready input exists:", MERGE_READY_PATH.exists(), MERGE_READY_PATH)
print("Category v1 output path:", CATEGORY_V1_OUTPUT_PATH)

Project root: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS
Merge-ready input exists: True /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/lida_storyweaver_merge_ready.json
Category v1 output path: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/story_categories_v1.json


## 1. Load Merge-Ready Dataset

In [2]:
with MERGE_READY_PATH.open(encoding="utf-8") as f:
    records = json.load(f)

df = pd.DataFrame(records)

print("Total records:", len(df))
display(df.head())

Total records: 464


,unified_id,source_dataset,source_id,title,text,word_count,reading_level,reading_level_key,original_category,source,source_url,author,illustrator,license_name,license_url,attribution
0,storyweaver_12758,storyweaver,12758,My Brother and Me,This is Samir. He is in class four. This is me...,100,Level 1,level_1,Activity Books,StoryWeaver,https://storyweaver.org.in/en/stories/12758-my...,Kanchan Bannerjee,Pallak Goswamy,None,None,None
1,storyweaver_25922,storyweaver,25922,What's Ameena Up To?,Baba is making coconut chutney. “Huh! Where di...,101,Level 1,level_1,Activity Books,StoryWeaver,https://storyweaver.org.in/en/stories/25922-wh...,Roopa Banerjee,Preetam Dhar,None,None,None
2,storyweaver_36201,storyweaver,36201,Fishing in My Village,"Last night, it rained so hard that water overf...",134,Level 1,level_1,Activity Books,StoryWeaver,https://storyweaver.org.in/en/stories/36201-fi...,Sanjana Khoobchandani,Nivong Sengsakoun,None,None,None
3,storyweaver_37230,storyweaver,37230,Tall and Short,Pa is getting my hat. Pa is tall. Ma is gettin...,113,Level 1,level_1,Activity Books,StoryWeaver,https://storyweaver.org.in/en/stories/37230-ta...,Celia Bolam,Seat Sopheap,None,None,None
4,storyweaver_14051,storyweaver,14051,The Best Thing Ever,Muzi loves to build things. He dreams of build...,183,Level 1,level_1,Activity Books,StoryWeaver,https://storyweaver.org.in/en/stories/14051-th...,Melissa Fagan ; Stefania Origgi,Lauren Nel,None,None,None


In [3]:
storyweaver_df = df[df["source_dataset"] == "storyweaver"].copy()

category_counts = (
    storyweaver_df["original_category"]
    .value_counts()
    .rename_axis("original_category")
    .reset_index(name="count")
    .sort_values("original_category")
)

display(category_counts)

,original_category,count
0,Activity Books,30
1,Adventure & Mystery,30
2,Animal Stories,30
14,Award Winning,22
11,Biographies,28
12,Classics,28
3,Family & Friends,30
9,Fantasy,29
4,Funny,30
5,Growing Up,30


## 2. Define Version 1 Category Mapping

These mappings are selected because the original StoryWeaver category names directly describe the intended broad class. Other StoryWeaver categories are intentionally excluded because they are not suitable to the themes.

In [4]:
CATEGORY_V1_MAPPING = {
    "Animal Stories": "Animals",
    "Family & Friends": "Daily Life",
    "Growing Up": "Daily Life",
    "STEM": "Science & Knowledge",
    "Non-fiction": "Science & Knowledge",
}

category_v1_df = storyweaver_df[
    storyweaver_df["original_category"].isin(CATEGORY_V1_MAPPING)
].copy()

category_v1_df["category_v1"] = category_v1_df["original_category"].map(CATEGORY_V1_MAPPING)

print("Version 1 records:", len(category_v1_df))
display(category_v1_df.head())

Version 1 records: 150


,unified_id,source_dataset,source_id,title,text,word_count,reading_level,reading_level_key,original_category,source,source_url,author,illustrator,license_name,license_url,attribution,category_v1
60,storyweaver_2,storyweaver,2,Smile Please!,He was ahead of the rabbit. He was ahead of th...,95,Level 1,level_1,Animal Stories,StoryWeaver,https://storyweaver.org.in/en/stories/2-smile-...,Manisha Chaudhry,Ajit Narayan,None,None,None,Animals
61,storyweaver_7,storyweaver,7,Fat King Thin Dog,This is a fat king. The fat king has a thin do...,68,Level 1,level_1,Animal Stories,StoryWeaver,https://storyweaver.org.in/en/stories/7-fat-ki...,Parismita,Parismita,None,None,None,Animals
62,storyweaver_332,storyweaver,332,"Bheema, the Sleepyhead","One day, Gauri, the cow, asked him, “Bheema, w...",263,Level 1,level_1,Animal Stories,StoryWeaver,https://storyweaver.org.in/en/stories/332-bhee...,Rajesh Khar,Shweta Mohapatra,None,None,None,Animals
63,storyweaver_12294,storyweaver,12294,The Race,Four friends want to have a race with their to...,55,Level 1,level_1,Animal Stories,StoryWeaver,https://storyweaver.org.in/en/stories/12294-th...,Kanchan Bannerjee,Kavya Singh; Natasha Mehra,None,None,None,Animals
64,storyweaver_1052,storyweaver,1052,Busy Ants,"Hello, I am the fourth one in the line. Can yo...",198,Level 1,level_1,Animal Stories,StoryWeaver,https://storyweaver.org.in/en/stories/1052-bus...,Kanchan Bannerjee,Deepa Balsavar,None,None,None,Animals


## 3. Check Version 1 Distribution

In [5]:
v1_source_category_counts = (
    category_v1_df.groupby(["category_v1", "original_category", "reading_level_key"])
    .size()
    .reset_index(name="count")
    .sort_values(["category_v1", "original_category", "reading_level_key"])
)

display(v1_source_category_counts)

v1_totals = (
    category_v1_df.groupby("category_v1")
    .size()
    .reset_index(name="count")
    .sort_values("category_v1")
)

display(v1_totals)

,category_v1,original_category,reading_level_key,count
0,Animals,Animal Stories,level_1,10
1,Animals,Animal Stories,level_2,10
2,Animals,Animal Stories,level_3,10
3,Daily Life,Family & Friends,level_1,10
4,Daily Life,Family & Friends,level_2,10
5,Daily Life,Family & Friends,level_3,10
6,Daily Life,Growing Up,level_1,10
7,Daily Life,Growing Up,level_2,10
8,Daily Life,Growing Up,level_3,10
9,Science & Knowledge,Non-fiction,level_1,10


,category_v1,count
0,Animals,30
1,Daily Life,60
2,Science & Knowledge,60


In [6]:
quality_by_category = (
    category_v1_df.groupby("category_v1")
    .agg(
        records=("unified_id", "count"),
        min_words=("word_count", "min"),
        median_words=("word_count", "median"),
        max_words=("word_count", "max"),
        very_short_texts=("word_count", lambda values: (values < 20).sum()),
    )
    .reset_index()
)

display(quality_by_category)

,category_v1,records,min_words,median_words,max_words,very_short_texts
0,Animals,30,55,358.0,913,0
1,Daily Life,60,1,292.0,1794,1
2,Science & Knowledge,60,1,210.0,1546,1


In [7]:
short_v1_texts = category_v1_df[category_v1_df["word_count"] < 20].sort_values(
    ["category_v1", "word_count", "title"]
)

print("Very short version 1 texts under 20 words:", len(short_v1_texts))
display(short_v1_texts[[
    "category_v1", "original_category", "source_id", "title", "word_count", "reading_level"
]])

Very short version 1 texts under 20 words: 2


,category_v1,original_category,source_id,title,word_count,reading_level
259,Daily Life,Growing Up,174103,The Girl Who Could Not Stop Laughing,1,Level 1
408,Science & Knowledge,STEM,173986,Angry Akku,1,Level 1


## 4. Preview Category Examples

In [8]:
example_preview = (
    category_v1_df.sort_values(["category_v1", "original_category", "reading_level_key", "title"])
    .groupby("category_v1")
    .head(8)
)

display(example_preview[[
    "category_v1", "original_category", "reading_level", "title", "word_count"
]])

,category_v1,original_category,reading_level,title,word_count
62,Animals,Animal Stories,Level 1,"Bheema, the Sleepyhead",263
64,Animals,Animal Stories,Level 1,Busy Ants,198
61,Animals,Animal Stories,Level 1,Fat King Thin Dog,68
66,Animals,Animal Stories,Level 1,How Many?,158
60,Animals,Animal Stories,Level 1,Smile Please!,95
67,Animals,Animal Stories,Level 1,The Greedy Mouse,231
68,Animals,Animal Stories,Level 1,The Lion's Howdah,171
65,Animals,Animal Stories,Level 1,The Mango Tree,73
174,Daily Life,Family & Friends,Level 1,Bunty and Bubbly,119
176,Daily Life,Family & Friends,Level 1,Colours on the Street,107


## 5. Save Version 1 Dataset

This output is the first labeled reference dataset for future classification. It contains only selected StoryWeaver records and excludes LIDA stories for now.

In [9]:
CATEGORY_V1_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

category_v1_output_columns = [
    "unified_id",
    "source_dataset",
    "source_id",
    "title",
    "text",
    "word_count",
    "reading_level",
    "reading_level_key",
    "original_category",
    "category_v1",
    "source",
    "source_url",
    "author",
    "illustrator",
]

category_v1_records = category_v1_df[category_v1_output_columns].to_dict(orient="records")

CATEGORY_V1_OUTPUT_PATH.write_text(
    json.dumps(category_v1_records, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved category version 1 dataset:", CATEGORY_V1_OUTPUT_PATH)
print("Records saved:", len(category_v1_records))

Saved category version 1 dataset: /Users/datong/Documents/5120/Nurodiversity inclusive design/data/TP10_DS/data/processed/story_categories_v1.json
Records saved: 150
